In [1]:
%%writefile /kaggle/working/03c_preprocess_mri_v3.py
# Praproses v3: 5 view x 32 irisan x 224 px, dipecah 2 bagian (batas output 19,5 GB)
import os, time
import numpy as np
import pandas as pd
import pydicom
import cv2
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

PART    = 1                           # 0 atau 1
NPART   = 2
BASE    = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
SPLIT   = "train"                     # "train" atau "test"
OUT_DIR = f"/kaggle/working/vol_{SPLIT}_v3_p{PART}"
META    = f"_meta_v3_p{PART}.csv"
IMG     = 224
DEPTH   = 32
LIMIT   = None                        # None = semua studi
VIEWS   = [("Sagittal", "fs"), ("Coronal", "fs"), ("Axial", "fs"),
           ("Sagittal", "nonfs"), ("Coronal", "nonfs")]


def choose_series(g, series_root):
    """fs   : utamakan fluid-sensitive + fat-sat
       nonfs: hanya series tanpa fat-sat, utamakan non-fluid-sensitive (PD/T1)
       Series yang sudah terpilih tidak dipakai dua kali."""
    g = g.copy()
    g["n"] = [len(os.listdir(os.path.join(series_root, s)))
              if os.path.isdir(os.path.join(series_root, s)) else 0
              for s in g["SeriesInstanceUID"]]
    g = g[g["n"] > 0]
    chosen, used = [], set()
    for plane, kind in VIEWS:
        c = g[(g["Anatomical_Plane"] == plane) & ~g["SeriesInstanceUID"].isin(used)].copy()
        if kind == "fs":
            c["score"] = c["Fluid_Sensitive"] * 2 + c["Fat_Suppression"]
        else:
            c = c[c["Fat_Suppression"] == 0].copy()
            c["score"] = 1 - c["Fluid_Sensitive"]
        c = c.sort_values(["score", "n"], ascending=False)
        s = c["SeriesInstanceUID"].iloc[0] if len(c) else None
        chosen.append(s)
        if s is not None:
            used.add(s)
    return chosen


def load_series(path):
    """Baca semua slice DICOM dan urutkan secara spasial (bukan nama file)."""
    slices = []
    for f in os.listdir(path):
        if not f.endswith(".dcm"):
            continue
        ds = pydicom.dcmread(os.path.join(path, f))
        try:
            pos = np.array(ds.ImagePositionPatient, dtype=float)
            ori = np.array(ds.ImageOrientationPatient, dtype=float)
            key = float(pos @ np.cross(ori[:3], ori[3:]))
        except Exception:
            key = float(getattr(ds, "InstanceNumber", 0))
        img = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = img.max() - img
        slices.append((key, img))
    slices.sort(key=lambda x: x[0])
    return [s[1] for s in slices]


def square_resize(img, size):
    h, w = img.shape
    m = max(h, w)
    canvas = np.zeros((m, m), np.float32)
    y, x = (m - h) // 2, (m - w) // 2
    canvas[y:y + h, x:x + w] = img
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)


def to_volume(slices):
    vol = np.stack([square_resize(s, IMG) for s in slices])
    fg = vol[vol > 0]
    lo, hi = np.percentile(fg if fg.size else vol, [0.5, 99.5])
    vol = np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1)
    idx = np.linspace(0, len(vol) - 1, DEPTH).round().astype(int)
    return (vol[idx] * 255).astype(np.uint8)


def process_study(study_uid, g):
    series_root = os.path.join(BASE, f"{SPLIT}_series", study_uid)
    chosen = choose_series(g, series_root)
    vol = np.zeros((len(VIEWS), DEPTH, IMG, IMG), np.uint8)
    mask = np.zeros(len(VIEWS), np.uint8)
    meta = []
    for i, ((plane, kind), s_uid) in enumerate(zip(VIEWS, chosen)):
        n, err = 0, ""
        if s_uid is not None:
            try:
                slices = load_series(os.path.join(series_root, s_uid))
                n = len(slices)
                if n:
                    vol[i] = to_volume(slices)
                    mask[i] = 1
            except Exception as e:
                err = f"{type(e).__name__}: {str(e)[:120]}"
        meta.append({"StudyInstanceUID": study_uid, "view": f"{plane}_{kind}",
                     "SeriesInstanceUID": s_uid, "n_slices": n, "ok": int(mask[i]), "error": err})
    np.savez_compressed(os.path.join(OUT_DIR, f"{study_uid}.npz"), vol=vol, mask=mask)
    return meta


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    series = pd.read_csv(os.path.join(BASE, f"{SPLIT}_series.csv"))
    groups = dict(tuple(series.groupby("StudyInstanceUID")))
    studies = sorted(groups)[PART::NPART] if SPLIT == "train" else sorted(groups)   # train dibagi 2
    studies = [s for s in studies if not os.path.exists(os.path.join(OUT_DIR, f"{s}.npz"))]
    if LIMIT:
        studies = studies[:LIMIT]
    print(f"PART {PART}/{NPART} | diproses: {len(studies)} studi, CPU: {os.cpu_count()}")

    t0, rows = time.time(), []
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
        futs = [ex.submit(process_study, s, groups[s]) for s in studies]
        for f in tqdm(as_completed(futs), total=len(futs)):
            rows.extend(f.result())

    meta_path = os.path.join(OUT_DIR, META)
    meta = pd.DataFrame(rows)
    if os.path.exists(meta_path):
        meta = pd.concat([pd.read_csv(meta_path), meta], ignore_index=True)
    meta.to_csv(meta_path, index=False)

    dt = time.time() - t0
    print(f"\nSelesai {len(studies)} studi dalam {dt/60:.1f} menit "
          f"({dt/max(len(studies),1):.2f} detik/studi)")
    print("View tersedia (%):")
    print((meta.groupby("view")["ok"].mean() * 100).round(1).to_string())
    print("Jumlah error:", (meta["error"].fillna("") != "").sum())
    size = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR))
    print(f"Ukuran folder output: {size/1e9:.2f} GB")


if __name__ == "__main__":
    main()

Writing /kaggle/working/03c_preprocess_mri_v3.py


In [2]:
!python /kaggle/working/03c_preprocess_mri_v3.py

PART 1/2 | diproses: 2203 studi, CPU: 4
100%|███████████████████████████████████████| 2203/2203 [33:42<00:00,  1.09it/s]

Selesai 2203 studi dalam 33.7 menit (0.92 detik/studi)
View tersedia (%):
view
Axial_fs          100.0
Coronal_fs        100.0
Coronal_nonfs      73.1
Sagittal_fs       100.0
Sagittal_nonfs     96.6
Jumlah error: 0
Ukuran folder output: 10.83 GB
